# Italia Corpus — Struttura e qualità del corpus normativo italiano

**Repo**: [ahmeabd/italia-corpus](https://github.com/ahmeabd/italia-corpus) \
**Fork Lab**: [dataciviclab/italia-corpus](https://github.com/dataciviclab/italia-corpus)

Italia Corpus è un progetto che converte la legislazione italiana (da Normattiva) in file Markdown.
Contiene oltre 280.000 atti normativi, dai regi decreti del Regno d'Italia ai DL del 2026.

Questo notebook analizza:
1. **Struttura del corpus** — quali collezioni ci sono, quanto pesano, cosa è vivo e cosa è storico
2. **Qualità del dataset** `normativa` — 20.711 atti deduplicati da 20 collezioni legislative, con metadati estratti

---
## 1. Struttura del corpus

L'analisi è fatta dal git tree object (non scarica i file, solo i metadati del repository).
Legge 288.279 file in circa 2 secondi.

In [1]:
import subprocess
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent

out = subprocess.run(
    ['git', 'ls-tree', '-r', 'HEAD'],
    capture_output=True, text=True, timeout=30,
    cwd=REPO
).stdout

files = []
for line in out.strip().split('\n'):
    if not line.strip():
        continue
    meta, path = line.split('\t')
    files.append({'path': path})

print(f'File totali: {len(files):,}')

File totali: 288,287


In [2]:
# Per tipo di file
exts = Counter()
for f in files:
    p = f['path']
    if '.' in p:
        ext = '.' + p.rsplit('.', 1)[1].lower()
    else:
        ext = '(nessuna)'
    exts[ext] += 1

print(f"{'Estensione':>20} | {'File':>10} | {'%':>6}")
print('-' * 42)
for ext, count in exts.most_common(10):
    print(f'{ext:>20} | {count:>10,} | {count*100/len(files):>5.1f}%')
print(f'\nFile .md: {exts.get(".md", 0):,} ({exts.get(".md", 0)*100/len(files):.1f}%)')
print(f'File .md\" (nome con virgoletta finale): {exts.get(".md\"", 0):,} — da verificare)')

          Estensione |       File |      %
------------------------------------------
                 .md |    283,748 |  98.4%
                .md" |      4,523 |   1.6%
                 .py |          5 |   0.0%
                .yml |          4 |   0.0%
           (nessuna) |          2 |   0.0%
            .parquet |          2 |   0.0%
          .gitignore |          1 |   0.0%
              .ipynb |          1 |   0.0%
               .toml |          1 |   0.0%

File .md: 283,748 (98.4%)
File .md" (nome con virgoletta finale): 4,523 — da verificare)


In [3]:
# Per collezione (directory di primo livello)
dirs = Counter()
for f in files:
    path = f['path']
    d = path.split('/')[0] if '/' in path else '(root)'
    dirs[d] += 1

total = len(files)
print(f"{'Collezione':>55} | {'File':>10} | {'%':>6} | {'Cumulativa':>10}")
print('-' * 86)
cum = 0
for d, count in dirs.most_common(30):
    cum += count
    print(f'{d:>55} | {count:>10,} | {count*100/total:>5.1f}% | {cum*100/total:>6.1f}%')

# Categorie
vive_nomi = ['DL e leggi di conversione', 'Decreti Legislativi', 'Leggi di ratifica',
        'Regolamenti ministeriali', 'Regolamenti governativi',
        'Atti di recepimento direttive UE', 'DPCM',
        'Decreti legislativi luogotenenziali', 'Leggi delega e relativi provvedimenti delegati',
        'Regolamenti di delegificazione', 'Testi Unici', 'Codici',
        'Leggi costituzionali', 'Leggi finanziarie e di bilancio',
        'Leggi contenenti deleghe', 'Atti di attuazione Regolamenti UE',
        'Leggi di delegazione europea']
vivi_file = sum(dirs[d] for d in vive_nomi if d in dirs)

storiche_nomi = ['Atti normativi abrogati (in originale)', 'Regi decreti', 'DPR']
stor_file = sum(dirs[d] for d in storiche_nomi if d in dirs)

print(f'\n---')
print(f'Collezioni di attualità normativa: {vivi_file:,} file ({vivi_file*100/total:.1f}%)')
print(f'Collezioni storiche (abrogati, regi decreti, DPR): {stor_file:,} file ({stor_file*100/total:.1f}%)')

                                             Collezione |       File |      % | Cumulativa
--------------------------------------------------------------------------------------
                 Atti normativi abrogati (in originale) |    121,761 |  42.2% |   42.2%
                                           Regi decreti |     89,125 |  30.9% |   73.2%
                                                    DPR |     47,251 |  16.4% |   89.5%
                              DL e leggi di conversione |      9,718 |   3.4% |   92.9%
                                    Decreti Legislativi |      2,891 |   1.0% |   93.9%
                                      Leggi di ratifica |      2,326 |   0.8% |   94.7%
                "Atti normativi abrogati (in originale) |      2,102 |   0.7% |   95.5%
                               Regolamenti ministeriali |      2,074 |   0.7% |   96.2%
                                Regolamenti governativi |      1,938 |   0.7% |   96.8%
                              

### Cosa significa

Il **90%** del corpus è storia archiviata: atti abrogati e norme pre-repubblicane.
Solo l'**8%** (~24.000 file) copre la produzione normativa corrente (leggi, DL, decreti, regolamenti, recepimento UE).

Per un uso mirato (ricerca, MCP, estrazione dati) ha senso concentrarsi sulle collezioni vive,
riducendo il volume da 475 MB a circa 50-70 MB.

---
## 2. Dataset normativa — Controllo qualità

Il dataset è generato da `lab_tools.extract()`: estrae metadati strutturati da tutte le 20 collezioni legislative attive.
Ogni file Markdown viene parsato per estrarre: tipo atto, data, numero, oggetto, entrata in vigore, riferimenti CELEX e ritardo.
Il dataset è deduplicato per filename (un atto può apparire in più collezioni — il campo `collezione` elenca tutte le occorrenze, separate da `;`).

In [4]:
import pandas as pd

df = pd.read_parquet(REPO / 'data' / 'derived' / 'normativa.parquet')
print(f'Atti analizzati: {len(df)}')
print(f'Colonne: {list(df.columns)}')
print(f'Periodo: {df.anno_atto.min()} - {df.anno_atto.max()}')

Atti analizzati: 20711
Colonne: ['collezione', 'filename', 'tipo', 'data', 'numero', 'oggetto', 'entrata_vigore', 'celex', 'anno_atto', 'anno_dir', 'ritardo']
Periodo: 1874 - 2026


In [5]:
# Completezza
print(f"{'Campo':>25} | {'Compilati':>10} | {'%':>6} | {'Vuoti':>10}")
print('-' * 56)
for col in df.columns:
    if col in ('ritardo',):
        nulls = (df[col].isna()).sum()
    elif col == 'anno_dir':
        nulls = (df[col] == 0).sum()
    elif col in ('celex', 'entrata_vigore'):
        nulls = (df[col] == '').sum()
    else:
        nulls = df[col].isna().sum()
    filled = len(df) - nulls
    print(f'{col:>25} | {filled:>10,} | {filled*100/len(df):>5.1f}% | {nulls:>10,}')

                    Campo |  Compilati |      % |      Vuoti
--------------------------------------------------------
               collezione |     20,711 | 100.0% |          0
                 filename |     20,711 | 100.0% |          0
                     tipo |     20,711 | 100.0% |          0
                     data |     20,711 | 100.0% |          0
                   numero |     20,711 | 100.0% |          0
                  oggetto |     20,711 | 100.0% |          0
           entrata_vigore |     11,543 |  55.7% |      9,168
                    celex |      2,867 |  13.8% |     17,844
                anno_atto |     20,711 | 100.0% |          0
                 anno_dir |      2,811 |  13.6% |     17,900
                  ritardo |      2,631 |  12.7% |     18,080


In [6]:
# Tipologia degli atti
print('Per tipo di atto:')
print(df['tipo'].value_counts().to_string())
print(f'\nI decreti legislativi rappresentano lo strumento principale di recepimento UE.')

Per tipo di atto:
tipo
LEGGE                                                10129
DECRETO LEGISLATIVO                                   2897
DECRETO DEL PRESIDENTE DELLA REPUBBLICA               2079
DECRETO                                               1833
DECRETO-LEGGE                                         1794
DECRETO LEGISLATIVO LUOGOTENENZIALE                   1213
DECRETO DEL PRESIDENTE DEL CONSIGLIO DEI MINISTRI      359
REGIO DECRETO                                          174
REGIO DECRETO LEGISLATIVO                              118
DECRETO MINISTERIALE                                    59
LEGGE COSTITUZIONALE                                    49
DECRETO LUOGOTENENZIALE                                  6
REGIO DECRETO-LEGGE                                      1

I decreti legislativi rappresentano lo strumento principale di recepimento UE.


In [7]:
# Ritardo di recepimento
r = df['ritardo'].dropna()
print(f'Ritardo (anni) tra pubblicazione della direttiva UE e atto di recepimento:')
print(f'  Minimo:     {r.min():.0f}')
print(f'  Massimo:    {r.max():.0f}')
print(f'  Medio:      {r.mean():.1f}')
print(f'  Mediano:    {r.median():.0f}')
print(f'  Oltre 5 anni: { (r > 5).sum() } atti ({ (r > 5).sum()/len(r)*100:.1f}%)')
print()

bins = list(range(0, 26, 2))
print(f'{"Intervallo":>12} | {"Atti":>6}')
print('-' * 22)
for i in range(len(bins)-1):
    cnt = ((r >= bins[i]) & (r < bins[i+1])).sum()
    if cnt:
        bar = '█' * cnt
        print(f'{bins[i]:>3}-{bins[i+1]-1:<3}  | {cnt:>6} {bar}')

Ritardo (anni) tra pubblicazione della direttiva UE e atto di recepimento:
  Minimo:     0
  Massimo:    42
  Medio:      5.2
  Mediano:    4
  Oltre 5 anni: 854 atti (32.5%)

  Intervallo |   Atti
----------------------
  0-1    |    392 ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  2-3    |    898 ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

In [8]:
# Copertura CELEX
con_celex = (df.celex != '').sum()
print(f'Atti con CELEX: {con_celex} / {len(df)} ({con_celex/len(df)*100:.1f}%)')
celex_counts = df[df.celex != '']['celex'].str.split(';').apply(len)
print(f'CELEX per atto: media {celex_counts.mean():.1f}, massimo {celex_counts.max()}')
print(f'CELEX totali: {celex_counts.sum():,}')

Atti con CELEX: 2867 / 20711 (13.8%)
CELEX per atto: media 11.6, massimo 876
CELEX totali: 33,393


In [9]:
# Andamento temporale
anni = df['anno_atto'].value_counts().sort_index()
print('Atti di recepimento per anno:')
print(f'{"Anno":>6} | {"Atti":>6}')
print('-' * 15)
for anno, cnt in anni.items():
    bar = '█' * min(cnt, 40)
    print(f'{anno:>6} | {cnt:>6} {bar}')

Atti di recepimento per anno:
  Anno |   Atti
---------------
  1874 |      6 ██████
  1876 |      1 █
  1877 |      2 ██
  1879 |      1 █
  1881 |      1 █
  1882 |      3 ███
  1883 |      4 ████
  1884 |      1 █
  1885 |      2 ██
  1886 |      1 █
  1888 |      3 ███
  1889 |      4 ████
  1892 |      2 ██
  1894 |      1 █
  1895 |      4 ████
  1896 |      4 ████
  1897 |      4 ████
  1898 |      4 ████
  1899 |      1 █
  1900 |      3 ███
  1901 |      1 █
  1902 |      5 █████
  1903 |      3 ███
  1904 |      2 ██
  1905 |      4 ████
  1906 |      2 ██
  1907 |      6 ██████
  1908 |      9 █████████
  1909 |      5 █████
  1910 |     12 ████████████
  1911 |      3 ███
  1913 |      2 ██
  1914 |      2 ██
  1916 |      2 ██
  1917 |      2 ██
  1918 |      1 █
  1919 |      3 ███
  1920 |      3 ███
  1921 |      2 ██
  1922 |      4 ████
  1923 |     16 ████████████████
  1924 |     10 ██████████
  1925 |    104 ████████████████████████████████████████
  1926 |    202 

In [10]:
# Entrata in vigore: gap tra pubblicazione e decorrenza
df_vig = df[(df.entrata_vigore != '') & (df.data != '')].copy()
if len(df_vig) > 0:
    df_vig['data_dt'] = pd.to_datetime(df_vig['data'], errors='coerce')
    df_vig['vigore_dt'] = pd.to_datetime(df_vig['entrata_vigore'], errors='coerce')
    df_vig['gap'] = (df_vig['vigore_dt'] - df_vig['data_dt']).dt.days
    gap = df_vig['gap'].dropna()
    print(f'Giorni tra pubblicazione ed entrata in vigore ({len(df_vig)} atti):')
    print(f'  Minimo:  {gap.min():.0f}')
    print(f'  Massimo: {gap.max():.0f}')
    print(f'  Medio:   {gap.mean():.1f}')
    print(f'  Mediano: {gap.median():.0f}')

Giorni tra pubblicazione ed entrata in vigore (11543 atti):
  Minimo:  -18161
  Massimo: 917
  Medio:   40.1
  Mediano: 34


---
## Osservazioni

### Sul corpus
- **288.279 file**, di cui 283.745 Markdown (98.4%)
- Il **90%** del volume è composto da atti abrogati, regi decreti e DPR — legislazione storica
- Le collezioni di attualità normativa sono circa **23.000 file**, un sottoinsieme gestibile per ricerca e indicizzazione
- 4.523 file presentano un nome con virgoletta finale (`.md"`) — possibile anomalia nella generazione upstream

### Sul dataset normativa (multi-collezione, deduplicato)
- **20.711 atti** unici, dal 1874 al 2026
- **2.867 (13.8%)** con riferimenti CELEX (33.393 CELEX totali)
- Ritardo medio: **5.2 anni**; massimo: 42 anni
- **854 atti** hanno un ritardo superiore a 5 anni (32.5% del campione con CELEX)
- L'entrata in vigore avviene in media **40 giorni** dopo la pubblicazione
- Le LEGGI sono lo strumento più frequente (48.9%), seguite da decreti legislativi (14.0%)

### Limiti noti
- L'86% degli atti non ha `anno_dir` (nessun CELEX L/R nel testo)
- Il 44% non ha `entrata_vigore` estraibile dal testo
- Il dataset copre le 20 collezioni attive del fork Lab, non l'intero corpus upstream (288k file)
- `ritardo` misura il gap con il CELEX di tipo L o R più recente, non è garantito che coincida con la specifica direttiva recepita